In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

PROCESSED_DATA_PATH = Path().resolve().parents[1] / "data" / "processed"

resultados_ = pd.read_parquet(PROCESSED_DATA_PATH / "resultados.parquet")

candidatos_ = pd.read_parquet(PROCESSED_DATA_PATH / "candidatos.parquet")

receitas_ = pd.read_parquet(PROCESSED_DATA_PATH / "receitas.parquet")

vagas_ = pd.read_parquet(PROCESSED_DATA_PATH / "vagas_deputado_federal.parquet")

qe = pd.read_csv(PROCESSED_DATA_PATH / "quociente_eleitoral.csv", sep=";")

In [2]:
resultados_1822 = (
    resultados_.loc[
        (resultados_["ano_eleicao"].isin([2018, 2022]))
        & (resultados_["ds_cargo"].str.upper() == "DEPUTADO FEDERAL")
    ]
    .groupby(
        [
            "ano_eleicao",
            "sg_uf",
            "sg_partido",
            "nr_candidato",
            "nm_candidato",
            "ds_sit_tot_turno",
        ],
        as_index=False,
    )["qt_votos_nominais"]
    .sum()
)

In [3]:
resultados_select = resultados_[
    [
        "ano_eleicao",
        "sg_uf",
        "ds_cargo",
        "sg_partido",
        "nr_candidato",
        "nm_candidato",
        "ds_sit_tot_turno",
        "qt_votos_nominais",
    ]
].reset_index(drop=True)

candidatos = candidatos_[
    ["ano_eleicao", "sg_uf", "nr_candidato", "nr_cpf_candidato"]
].drop_duplicates(subset=["ano_eleicao", "sg_uf", "nr_candidato"], keep="first")

resultados_select = resultados_select.merge(
    candidatos, on=["ano_eleicao", "sg_uf", "nr_candidato"], how="left"
)

txt_eleitos = [
    "ELEITO",
    "ELEITO POR MÉDIA",
    "MÉDIA",
    "ELEITO POR QP",
]

resultados_select["eleito"] = np.where(
    resultados_select["ds_sit_tot_turno"].isin(txt_eleitos), 1, 0
)

# resultados_select = resultados_select.drop(columns=["ds_sit_tot_turno"])

resultados_select['ds_cargo'] = resultados_select['ds_cargo'].str.upper()

In [4]:
print(len(resultados_1822))
resultados_1822_cand = resultados_1822.merge(
    candidatos, on=["ano_eleicao", "sg_uf", "nr_candidato"], how="left"
)
print(len(resultados_1822_cand))

17305
17305


In [5]:
def _identificar_historico_eleitoral(
    resultados_cpf: pd.DataFrame, ano_eleicao_interesse: int, ano_limite: int
) -> pd.DataFrame:
    # cria dummies de eleitos por cargo
    cargos = [
        "PREFEITO",
        "VEREADOR",
        "DEPUTADO ESTADUAL",
        "DEPUTADO FEDERAL",
        "GOVERNADOR",
        "SENADOR",
        "DEPUTADO DISTRITAL",
    ]

    for cargo in cargos:
        clean_cargo = cargo.lower().replace(" ", "_")
        resultados_cpf[f"dummy_{clean_cargo}"] = (
            (resultados_cpf["ds_cargo"] == cargo) & (resultados_cpf["eleito"] == 1)
        ).astype(int)

    # para identificar o historico eleitoral, pegar resultados até a eleição anterior a de interesse
    # se vamos olhar as eleições de 2018, pegamos os resultados das eleições até 2016
    resultados_cpf_eleicao_anterior = resultados_cpf.query(
        f"ano_eleicao <= {ano_limite}"
    )

    # identificando eleicoes por cpf
    colunas_final = resultados_cpf_eleicao_anterior.filter(
        like="dummy_"
    ).columns.tolist() + ["nr_cpf_candidato"]

    # tabela com o cpf do candidato e o n de eleições para cada cargo até a eleição anterior a de interesse
    cpf_dummies = (
        resultados_cpf_eleicao_anterior[colunas_final]
        .groupby("nr_cpf_candidato", as_index=False)
        .sum()  # soma as dummies, se o cpf foi eleito mais de uma vez para o mesmo cargo, vai somar 1 cada eleição
    )

    # consolidando dep estadual + dep distrital (brasilia)
    cpf_dummies["dummy_deputado_estadual"] = (
        cpf_dummies["dummy_deputado_estadual"] + cpf_dummies["dummy_deputado_distrital"]
    )
    cpf_dummies = cpf_dummies.drop("dummy_deputado_distrital", axis=1)

    # renomeando colunas
    cpf_dummies = cpf_dummies.rename(
        columns={
            "dummy_prefeito": "n_eleicoes_prefeito",
            "dummy_vereador": "n_eleicoes_vereador",
            "dummy_deputado_estadual": "n_eleicoes_deputado_estadual",
            "dummy_deputado_federal": "n_eleicoes_deputado_federal",
            "dummy_governador": "n_eleicoes_governador",
            "dummy_senador": "n_eleicoes_senador",
        }
    )

    cpf_dummies["ano_eleicao"] = ano_eleicao_interesse

    return cpf_dummies


def consolidar_historico(resultados_select):
    hist18 = _identificar_historico_eleitoral(
        resultados_select, ano_eleicao_interesse=2018, ano_limite=2016
    )
    hist22 = _identificar_historico_eleitoral(
        resultados_select, ano_eleicao_interesse=2022, ano_limite=2020
    )
    historico = pd.concat([hist18, hist22], axis=0).reset_index(drop=True)
    return historico


historico = consolidar_historico(resultados_select)

In [6]:
resultados_1822_cand_historico = resultados_1822_cand.merge(
    historico, on=["ano_eleicao", "nr_cpf_candidato"], how="left"
)

In [7]:
receitas_df = receitas_.loc[receitas_["ds_cargo"] == 'DEPUTADO FEDERAL'].copy()

receitas_df["ds_origem_receita"] = np.where(
    receitas_df["ds_origem_receita"] != "Recursos de partido político",
    "vr_receita_outros",
    "vr_receita_recursos_partidos",
)

In [8]:
receitas_df = receitas_df.groupby(["ano_eleicao", "sg_uf", "nr_candidato", "ds_origem_receita"], as_index=False)["vr_receita"].sum()

receitas_df = receitas_df.pivot(
    index=["ano_eleicao", "sg_uf", "nr_candidato"],
    columns="ds_origem_receita",
    values="vr_receita",
).reset_index()

In [9]:
receitas_df['nr_candidato'] = receitas_df['nr_candidato'].astype(str)

In [10]:
print(len(resultados_1822_cand_historico))
resultados_1822_cand_historico_receita = resultados_1822_cand_historico.merge(
    receitas_df, on=["ano_eleicao", "sg_uf", "nr_candidato"], how="left"
)
print(len(resultados_1822_cand_historico_receita))

17305
17305


In [11]:
print(len(resultados_1822_cand_historico_receita))
resultados_1822_cand_historico_receita_qe = (
    resultados_1822_cand_historico_receita.merge(
        qe, on=["ano_eleicao", "sg_uf"], how="left"
    )
)
print(len(resultados_1822_cand_historico_receita_qe))

17305
17305


Adicionar analise temporal

In [12]:
raw_path = Path().resolve().parent.parent / "data" / "raw"


def calcular_dias_primeira_transferencia18():
    def processar_receitas_partido_2018():
        receitas18 = pd.read_csv(
            raw_path / "finanças" / "receitas_candidatos_2018_BRASIL.csv",
            sep=";",
            encoding="latin1",
            dtype={
                "NR_CANDIDATO": str,
                "ANO_ELEICAO": str,
                "SG_PARTIDO": str,
                "SG_UF": str,
                "NR_CANDIDATO": str,
            },
        )
        receitas18.columns = receitas18.columns.str.strip().str.lower()
        receitas18["dt_receita"] = pd.to_datetime(
            receitas18["dt_receita"], dayfirst=True
        )
        receitas18["vr_receita"] = (
            receitas18["vr_receita"].str.replace(",", ".").astype(float)
        )
        receitas18["vr_receita_mi"] = receitas18["vr_receita"] / 1_000_000
        receitas18_partido_df = receitas18.loc[
            (receitas18["ds_cargo"] == "Deputado Federal")
            & (receitas18["ds_origem_receita"] == "Recursos de partido político")
            & (receitas18["dt_receita"] >= "2018-08-16")
            & (receitas18["dt_receita"] <= "2018-10-07")
        ].copy()
        return receitas18_partido_df

    receitas18_partido_df = processar_receitas_partido_2018()
    df_primeira_transf = receitas18_partido_df.groupby(
        ["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"], as_index=False
    )["dt_receita"].min()

    # diferença em dias em relação à data fixa de início
    df_primeira_transf["dias_desde_inicio"] = (
        df_primeira_transf["dt_receita"] - pd.Timestamp("2018-08-16")
    ).dt.days

    return df_primeira_transf


def calcular_dias_primeira_transferencia22():
    def processar_receitas_partido_2022():
        receitas22 = pd.read_csv(
            raw_path / "finanças" / "receitas_candidatos_2022_BRASIL.csv",
            sep=";",
            encoding="latin1",
            dtype={
                "NR_CANDIDATO": str,
                "ANO_ELEICAO": str,
                "SG_PARTIDO": str,
                "SG_UF": str,
                "NR_CANDIDATO": str,
            },
        )
        receitas22.columns = receitas22.columns.str.strip().str.lower()
        receitas22["dt_receita"] = pd.to_datetime(
            receitas22["dt_receita"], dayfirst=True
        )
        receitas22["vr_receita"] = (
            receitas22["vr_receita"].str.replace(",", ".").astype(float)
        )
        receitas22["vr_receita_mi"] = receitas22["vr_receita"] / 1_000_000
        receitas22_partido_df = receitas22.loc[
            (receitas22["ds_cargo"] == "Deputado Federal")
            & (receitas22["ds_origem_receita"] == "Recursos de partido político")
            & (receitas22["dt_receita"] >= "2022-08-16")
            & (receitas22["dt_receita"] <= "2022-10-02")
        ].copy()
        return receitas22_partido_df

    receitas22_partido_df = processar_receitas_partido_2022()
    df_primeira_transf = receitas22_partido_df.groupby(
        ["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"], as_index=False
    )["dt_receita"].min()

    # diferença em dias em relação à data fixa de início
    df_primeira_transf["dias_desde_inicio"] = (
        df_primeira_transf["dt_receita"] - pd.Timestamp("2022-08-16")
    ).dt.days

    return df_primeira_transf


df_primeira_transf18 = calcular_dias_primeira_transferencia18()
df_primeira_transf22 = calcular_dias_primeira_transferencia22()

df_primeira_transf = pd.concat(
    [df_primeira_transf18, df_primeira_transf22], axis=0
).reset_index(drop=True)

df_primeira_transf['ano_eleicao'] = df_primeira_transf['ano_eleicao'].astype(int)

print(len(resultados_1822_cand_historico_receita_qe))
resultados_1822_cand_historico_receita_tempo = (
    resultados_1822_cand_historico_receita_qe.merge(
        df_primeira_transf,
        on=["ano_eleicao", "sg_uf", "sg_partido", "nr_candidato"],
        how="left",
    )
)
print(len(resultados_1822_cand_historico_receita_tempo))

C:\Users\yuri.taba\AppData\Local\Temp\ipykernel_11688\3090627803.py:49: DtypeWarning: Columns (39,52) have mixed types. Specify dtype option on import or set low_memory=False.
  receitas22 = pd.read_csv(


17305
17305


In [13]:
cols_vitoria_eleitoral = resultados_1822_cand_historico_receita_tempo.filter(
    like="n_eleicoes"
).columns

for col in cols_vitoria_eleitoral:
    resultados_1822_cand_historico_receita_tempo[col] = np.where(
        resultados_1822_cand_historico_receita_tempo['nr_cpf_candidato'] == "-4", np.nan, 
        resultados_1822_cand_historico_receita_tempo[col]
    )

incluir:
- prop votos nominais lag
- qt vagas

In [14]:
resultados_1822_cand_historico_receita_tempo = (
    resultados_1822_cand_historico_receita_tempo.merge(
        vagas_.drop(columns=["ds_cargo"]), on=["ano_eleicao", "sg_uf"], how="left"
    )
)

In [15]:
# prop_votos_nominais_lag: share of intraparty list nominal votes in the prior election
# 2014 results → used for 2018 candidates; 2018 results → used for 2022 candidates

dep_fed_resultados_lag = resultados_select[
    (resultados_select["ds_cargo"] == "DEPUTADO FEDERAL")
    & (resultados_select["ano_eleicao"].isin([2014, 2018]))
].copy()

# total votes per list (partido × uf × year)
total_votos_lista = (
    dep_fed_resultados_lag
    .groupby(["ano_eleicao", "sg_uf", "sg_partido"])["qt_votos_nominais"]
    .sum()
    .rename("qt_votos_lista")
    .reset_index()
)

dep_fed_resultados_lag = dep_fed_resultados_lag.merge(
    total_votos_lista, on=["ano_eleicao", "sg_uf", "sg_partido"], how="left"
)
dep_fed_resultados_lag["prop_votos_nominais"] = (
    dep_fed_resultados_lag["qt_votos_nominais"] / dep_fed_resultados_lag["qt_votos_lista"]
)

# keep the best (max) share per CPF × list in case of duplicates, then rename year to target year
votos_lag = (
    dep_fed_resultados_lag
    .groupby(["ano_eleicao", "sg_uf", "sg_partido", "nr_cpf_candidato"], as_index=False)
    ["prop_votos_nominais"]
    .sum()
)

# shift: 2014 → 2018, 2018 → 2022
votos_lag["ano_eleicao"] = votos_lag["ano_eleicao"] + 4
votos_lag = votos_lag.rename(columns={"prop_votos_nominais": "prop_votos_nominais_lag"})

resultados_1822_cand_historico_receita_tempo = resultados_1822_cand_historico_receita_tempo.merge(
    votos_lag,
    on=["ano_eleicao", "sg_uf", "sg_partido", "nr_cpf_candidato"],
    how="left",
)

# first-timers (valid CPF but no prior record in same list) → 0
mask_valid_cpf = resultados_1822_cand_historico_receita_tempo["nr_cpf_candidato"] != "-4"
resultados_1822_cand_historico_receita_tempo.loc[
    mask_valid_cpf & resultados_1822_cand_historico_receita_tempo["prop_votos_nominais_lag"].isna(),
    "prop_votos_nominais_lag",
] = 0.0
# candidates with unknown CPF ("-4") remain NaN, consistent with n_eleicoes_* convention

In [16]:
resultados_

,ano_eleicao,nr_turno,sg_uf,cd_municipio,nm_municipio,ds_cargo,nr_candidato,nm_candidato,sg_partido,ds_composicao_coligacao,ds_sit_tot_turno,qt_votos_nominais,turnos_eleicao
0,2000,1,AC,01007,BUJARI,Prefeito,12,SILVINO ANTONIO DE OLIVEIRA,PDT,PDT,NÃO ELEITO,0,[1]
1,2000,1,AC,01007,BUJARI,Prefeito,13,EVANDRO OLIVEIRA CARDOSO,PT,PT,NÃO ELEITO,0,[1]
2,2000,1,AC,01007,BUJARI,Prefeito,15,JOÃO EDVALDO TELES DE LIMA,PMDB,PMDB,ELEITO,0,[1]
3,2000,1,AC,01007,BUJARI,Vereador,11610,FRANCISCO DA SILVA,PPB,PPB,SUPLENTE,0,[1]
4,2000,1,AC,01007,BUJARI,Vereador,11611,RAIMUNDO BEZERRA DE LIMA,PPB,PPB,SUPLENTE,0,[1]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2592390,2022,2,SC,NA,None,Governador,22,JORGINHO DOS SANTOS MELLO,PL,PL,ELEITO,2983949,[1 2]
2592391,2022,2,SE,NA,None,Governador,13,ROGERIO CARVALHO SANTOS,PT,Federação Brasil da Esperança - FE BRASIL(PT/P...,NÃO ELEITO,582940,[1 2]
2592392,2022,2,SE,NA,None,Governador,55,FÁBIO CRUZ MITIDIERI,PSD,PDT / PSC / UNIÃO / REPUBLICANOS / PP / PSD / ...,ELEITO,623851,[1 2]
2592393,2022,2,SP,NA,None,Governador,10,TARCISIO GOMES DE FREITAS,REPUBLICANOS,REPUBLICANOS / PL / PSD / PTB / PSC / PMN,ELEITO,13480643,[1 2]


In [ ]:
# colunas mínimas para a flag histórica
cols_res = [
    "ano_eleicao", "sg_uf", "cd_municipio", "nr_candidato",
    "ds_cargo", "nr_turno", "qt_votos_nominais"
]

# reduz o volume antes do merge
res_base = resultados_.loc[
    resultados_["ds_cargo"].str.upper().isin(
        ["DEPUTADO FEDERAL", "DEPUTADO ESTADUAL", "GOVERNADOR", "SENADOR", "PREFEITO"]
    )
    & (resultados_["nr_turno"] == 1),
    cols_res
].copy()

# normaliza tipos de chave
for c in ["ano_eleicao", "sg_uf", "nr_candidato"]:
    res_base[c] = res_base[c].astype(str)

cand_cpf = candidatos_[["ano_eleicao", "sg_uf", "nr_candidato", "nr_cpf_candidato"]].copy()
for c in ["ano_eleicao", "sg_uf", "nr_candidato"]:
    cand_cpf[c] = cand_cpf[c].astype(str)

# garante relação m:1 (evita explosão de linhas)
cand_cpf = cand_cpf.drop_duplicates(
    subset=["ano_eleicao", "sg_uf", "nr_candidato"],
    keep="first"
)

# merge seguro
res_cpf = res_base.merge(
    cand_cpf,
    on=["ano_eleicao", "sg_uf", "nr_candidato"],
    how="left",
    validate="m:1",
    sort=False,
    copy=False
)

# opcional: reduzir memória de strings repetidas
res_cpf["ds_cargo"] = res_cpf["ds_cargo"].astype("category")
res_cpf["sg_uf"] = res_cpf["sg_uf"].astype("category")

In [18]:
# ---------------------------------------------------------------------------
# alcancou_10pct_qe_hist — candidato atingiu >=10% QE em eleição anterior
# (cargo: Dep. Federal, Dep. Estadual, Governador, Senador, Prefeito)
# Vinculado via CPF; construído aqui para que rrd_df já carregue a flag pronta.
# ---------------------------------------------------------------------------

CARGOS_FORTE = ['DEPUTADO FEDERAL', 'DEPUTADO ESTADUAL', 'GOVERNADOR', 'SENADOR', 'PREFEITO']

# Passo 1: resultados históricos com CPF e município
# res_cpf = pd.read_csv(
#     PROCESSED_DATA_PATH / "resultados_cpf.csv",
#     encoding='latin1', sep=';',
# )

res_cpf.columns = res_cpf.columns.str.lower()
res_cpf['ds_cargo_up'] = res_cpf['ds_cargo'].str.upper()

hist = res_cpf[
    res_cpf['ds_cargo_up'].isin(CARGOS_FORTE) &
    (res_cpf['nr_turno'] == 1) &
    (res_cpf['nr_cpf_candidato'].astype(str) != '-4')
].copy()
hist['nr_cpf_candidato'] = hist['nr_cpf_candidato'].astype(str)

# Passo 2: vagas por cargo/UF/ano (Dep. Federal, Dep. Estadual, Senador variam)
_vagas_anos = [1998, 2002, 2006, 2010, 2014, 2018, 2022]
_vagas_list = []
for _ano in _vagas_anos:
    _f = PROCESSED_DATA_PATH.parent / f"raw/vagas/consulta_vagas_{_ano}_BRASIL.csv"
    _v = pd.read_csv(_f, sep=';', encoding='latin1')
    _v.columns = _v.columns.str.lower()
    if 'qt_vagas' in _v.columns:
        _v = _v.rename(columns={'qt_vagas': 'qt_vaga'})
    _v = _v[['ano_eleicao', 'sg_uf', 'ds_cargo', 'qt_vaga']]
    _vagas_list.append(_v)
vagas_all_cargos = pd.concat(_vagas_list, ignore_index=True)
vagas_all_cargos['ds_cargo'] = vagas_all_cargos['ds_cargo'].str.upper()
vagas_all_cargos = vagas_all_cargos.drop_duplicates(['ano_eleicao', 'sg_uf', 'ds_cargo'])
vagas_lkp = vagas_all_cargos[
    vagas_all_cargos['ds_cargo'].isin(['DEPUTADO FEDERAL', 'DEPUTADO ESTADUAL', 'SENADOR'])
].rename(columns={'ds_cargo': 'ds_cargo_up'})

# Passo 3a: cargos de nível UF
_uf_cargos = ['DEPUTADO FEDERAL', 'DEPUTADO ESTADUAL', 'GOVERNADOR', 'SENADOR']
hist_uf = hist[hist['ds_cargo_up'].isin(_uf_cargos)].copy()

votos_cand_uf = (
    hist_uf.groupby(['ano_eleicao', 'sg_uf', 'ds_cargo_up', 'nr_cpf_candidato'])['qt_votos_nominais']
    .sum().reset_index()
)
total_votos_uf = (
    hist_uf.groupby(['ano_eleicao', 'sg_uf', 'ds_cargo_up'])['qt_votos_nominais']
    .sum().reset_index().rename(columns={'qt_votos_nominais': 'total_votos'})
)

votos_cand_uf['ano_eleicao'] = votos_cand_uf['ano_eleicao'].astype('int64')
total_votos_uf['ano_eleicao'] = total_votos_uf['ano_eleicao'].astype('int64')


votos_cand_uf = votos_cand_uf.merge(total_votos_uf, on=['ano_eleicao', 'sg_uf', 'ds_cargo_up'])
votos_cand_uf = votos_cand_uf.merge(
    vagas_lkp[['ano_eleicao', 'sg_uf', 'ds_cargo_up', 'qt_vaga']],
    on=['ano_eleicao', 'sg_uf', 'ds_cargo_up'], how='left'
)
votos_cand_uf['qt_vaga'] = votos_cand_uf['qt_vaga'].fillna(1)  # Governador = 1

# Passo 3b: Prefeito (nível município, sempre 1 vaga)
hist_mun = hist[hist['ds_cargo_up'] == 'PREFEITO'].copy()
votos_cand_mun = (
    hist_mun.groupby(['ano_eleicao', 'cd_municipio', 'ds_cargo_up', 'nr_cpf_candidato'])['qt_votos_nominais']
    .sum().reset_index()
)
total_votos_mun = (
    hist_mun.groupby(['ano_eleicao', 'cd_municipio', 'ds_cargo_up'])['qt_votos_nominais']
    .sum().reset_index().rename(columns={'qt_votos_nominais': 'total_votos'})
)
votos_cand_mun = votos_cand_mun.merge(total_votos_mun, on=['ano_eleicao', 'cd_municipio', 'ds_cargo_up'])
votos_cand_mun['qt_vaga'] = 1

# Passo 4: pct_qe = votos_candidato / (total_votos / n_vagas)
all_votos_qe = pd.concat([
    votos_cand_uf[['ano_eleicao', 'ds_cargo_up', 'nr_cpf_candidato', 'qt_votos_nominais', 'total_votos', 'qt_vaga']],
    votos_cand_mun[['ano_eleicao', 'ds_cargo_up', 'nr_cpf_candidato', 'qt_votos_nominais', 'total_votos', 'qt_vaga']],
], ignore_index=True)
all_votos_qe['qe_cargo'] = all_votos_qe['total_votos'] / all_votos_qe['qt_vaga']
all_votos_qe['pct_qe'] = np.where(
    all_votos_qe['qe_cargo'] > 0,
    all_votos_qe['qt_votos_nominais'] / all_votos_qe['qe_cargo'],
    0
)

# Passo 5: flag por ano-alvo (somente eleições *anteriores* ao ano-alvo)
resultados_1822_cand_historico_receita_tempo['nr_cpf_candidato'] = (
    resultados_1822_cand_historico_receita_tempo['nr_cpf_candidato'].astype(str)
)
resultados_1822_cand_historico_receita_tempo['alcancou_10pct_qe_hist'] = False



C:\Users\yuri.taba\AppData\Local\Temp\ipykernel_11688\3768052617.py:10: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  res_cpf = pd.read_csv(


In [19]:
all_votos_qe['ano_eleicao'] = all_votos_qe['ano_eleicao'].astype(int)

for _target_year in [2018, 2022]:
    _hist_antes = all_votos_qe[all_votos_qe['ano_eleicao'] < _target_year]
    _cpfs_fortes = set(_hist_antes.loc[_hist_antes['pct_qe'] >= 0.10, 'nr_cpf_candidato'])
    _mask = (
        (resultados_1822_cand_historico_receita_tempo['ano_eleicao'] == _target_year) &
        resultados_1822_cand_historico_receita_tempo['nr_cpf_candidato'].isin(_cpfs_fortes)
    )
    resultados_1822_cand_historico_receita_tempo.loc[_mask, 'alcancou_10pct_qe_hist'] = True

# CPF inválido ("-4") → NaN (consistente com convenção de n_eleicoes_*)
_mask_inv = resultados_1822_cand_historico_receita_tempo['nr_cpf_candidato'] == '-4'
resultados_1822_cand_historico_receita_tempo.loc[_mask_inv, 'alcancou_10pct_qe_hist'] = np.nan

print("alcancou_10pct_qe_hist adicionado à rrd_df.")
print(resultados_1822_cand_historico_receita_tempo
      .groupby('ano_eleicao')['alcancou_10pct_qe_hist']
      .value_counts())

alcancou_10pct_qe_hist adicionado à rrd_df.
ano_eleicao  alcancou_10pct_qe_hist
2018         False                     7230
             True                       396
2022         False                     9267
             True                       405
Name: count, dtype: int64


C:\Users\yuri.taba\AppData\Local\Temp\ipykernel_11688\3965639195.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  resultados_1822_cand_historico_receita_tempo.loc[_mask_inv, 'alcancou_10pct_qe_hist'] = np.nan


In [20]:
total_votos_uf.dtypes

ano_eleicao     int64
sg_uf          object
ds_cargo_up    object
total_votos     int64
dtype: object

In [21]:
resultados_1822_cand_historico_receita_tempo["10pct_qe_eleicao_atual"] = (
    resultados_1822_cand_historico_receita_tempo["qt_votos_nominais"]
    / resultados_1822_cand_historico_receita_tempo["qe"]
) > 0.10

TXT_ELEITOS = ["ELEITO", "ELEITO POR MÉDIA", "MÉDIA", "ELEITO POR QP"]

resultados_1822_cand_historico_receita_tempo["eleito"] = np.where(resultados_1822_cand_historico_receita_tempo["ds_sit_tot_turno"].isin(TXT_ELEITOS), 1, 0)

resultados_1822_cand_historico_receita_tempo["10pct_qe_eleicao_atual"].value_counts()

10pct_qe_eleicao_atual
False    15085
True      2220
Name: count, dtype: int64

In [22]:
if len(resultados_1822_cand_historico_receita_tempo[
    resultados_1822_cand_historico_receita_tempo["ds_sit_tot_turno"].isin(
        ["ELEITO POR QP", "ELEITO POR MÉDIA"]
    )
]) == 513*2:
    print("Deu bom!")
    resultados_1822_cand_historico_receita_tempo.to_parquet(
        PROCESSED_DATA_PATH / "rrd_df_csvcpf.parquet", index=False
    )

Deu bom!
